In [0]:
%run ../config/config-env

In [0]:
from pyspark.sql import functions as F


In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.drivers'
silver_table = f'{catalog_name}.{silver_schema}.drivers'

In [0]:
driver_df = spark.read.table(bronze_table)
driver_df.display()

In [0]:
driver_selected_df = driver_df.select(
    F.col('dateOfBirth'),
    F.col('driverId'),
    F.col('name.givenName'),
    F.col('name.FamilyName'),
    F.col('nationality'),
    F.col('ingestion_timestamp'),
    F.col('source_file')
)

In [0]:
display(driver_selected_df)

In [0]:
driver_name_df=driver_selected_df.withColumn('FamilyName', F.lit(F.col('DriverId')))

In [0]:
display(driver_name_df)

In [0]:
driver_named_df = driver_name_df.withColumn(
        'driver_name', F.concat_ws(' ', 'givenName', 'FamilyName')
        
        )
display(driver_named_df)

In [0]:
driver_dropped_df = driver_named_df.drop('givenName', 'FamilyName')
display(driver_dropped_df)

In [0]:
driver_selected_df = driver_dropped_df.select(
    F.col('dateOfBirth'),
    F.col('driverId'),
    F.col('driver_name'),
    F.col('nationality'),
    F.col('ingestion_timestamp'),
    F.col('source_file')
)
display(driver_selected_df)

In [0]:
driver_deduplicated_df = driver_selected_df.dropDuplicates(['driverId'])
display(driver_deduplicated_df)

In [0]:
driver_final_df = driver_deduplicated_df.withColumnsRenamed({
    'dateOfBirth': 'date_of_birth', 'driverId':'driver_id'
})

In [0]:
driver_finally_df = driver_final_df.withColumn('nationality', F.initcap(F.col('nationality')))

In [0]:
display(driver_finally_df)

In [0]:
(
    driver_finally_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)